# Семинар 12 - криптография

Сначала поговорим немного о сетях. Вспомним, что у нас были протоколы HTTP (hyper text transfer protocol) и HTTPS (... + secure). HTTPS в качестве метода шифрования использует протокол TLS (SSL). Примерно к 2017 году более 50% сайтов стали использовать HTTPS по умолчанию (или автоматически редиректить http -> https). Чтобы понять, почему это очень важно, можно использовать программы вроде Wireshark. Они "сниффают" весь интернет трафик на заданном интерфейсе. Используя их можно увидеть, что все данные, передающиеся по HTTP, открыты для чтения. Следовательно, злоумышленник, имеющий возможность прослушивать ваш трафик, может читать его.

## Хэши

Цель хеш-функции - конвертировать произвольную последовательность бит (строку) в последовательность бит фиксированной длины (хеш-значение). При этом делать это таким образом, чтобы восстановить по хеш-значению исходную строку было в некотором смысле сложно.

Несколько комментариев к определению:
* У хеш-функции существуют коллизии, то есть разные входы, маппящиеся в один и тот же выход
* Мы предъявляем требование, что по выходу хеш-функции должно быть сложно получить *любой* вход, ему соответствующий (не обязательно именно тот, который был использован изначально)

В криптографии подобные функции называются *односторонними*, вопрос их существования остаётся [открытым](https://en.wikipedia.org/wiki/One-way_function). 
> Из существования односторонних функций следует P ≠ NP. Обратная импликация неверна

Важно понимать, что хеш-функции не являются способами шифрования данных, они решают другие задачи.

Применение хэшей, которое нас сейчас интересует, это хранение паролей и других секретов. Хранение паролей в открытом виде имеет несколько проблем:
* Возможность доступа сотрудниками, имеющими доступ к хранилищу
* При компрометации хранилища все пароли станут известны атакующему

Чтобы решить эти проблемы, можно хранить не сами пароли, а их хэш. Тогда для проверки корректности пароля нужно сравнить хэши. Из этого вытекает неприятное следствие: существует бесконечено много неверных паролей, имеющих такой же хеш, как оригинальный пароль. Эта проблема полностью не решается, но использование "надежных" хеш-функций позволяет не переживать о ней. 

Для упомянутых целей важно использовать криптографически-стойкие хеш функции, то есть те, для которых сложно находить коллизии, и для которых нет достаточно эффективных способов обращения. 
Например, SHA-1 или MD5 на данный момент считаются "взломанными" хеш-функциями, поэтому они не должны использоваться в криптографических целях.

У предложенного подхода, однако, есть еще одна существенная проблема:

<details>
  <summary>Проблема</summary>
  
  Если пользователи используют одинаковые пароли, у них будут совпадать хеши. Это, например, позволяет атакующему компрометировать пароль уязвимой жертвы, хеш пароля которой совпадает с хешем пароля желаемой цели.
  
</details>

## Соль

Для решения проблемы выше используется следующая идея: считается хеш не от самого пароля, а от пароля, смешанного со случайной строкой бит (она называется соль). Эти биты случайно генерируются для каждого нового пароля. Это позволяет достичь того, что для одинаковых паролей будут сохранены разные значения хеш-функции.

Соль хранится в открытом виде, так же как и значение хеш-функции. Для проверки корректности пароля, он смешивается с солью и высчитывается хеш. 

Хранение соли в открытом виде не помогает злоумышленнику, потому что хеш-функция трудно обратима и частичное знание входа не помогает в обращении.

In [1]:
!echo "user=Vasya password_hash=$(echo -n 3.1415-2.718182 | openssl sha256 -r)"
!echo "user=Petya password_hash=$(echo -n sfkjvdjkth | openssl sha256 -r)"
!echo "user=admin password_hash=$(echo -n 3.1415-2.718182 | openssl sha256 -r)"

user=Vasya password_hash=a6f62b5131e63fac2e6f1be3e443a12e58e2c5fea002df0924f58eeefb7e81a9 *stdin
user=Petya password_hash=eea9d8bec1b74e88807bf93f3a0e095df6543b83d46550456d7f8d2139c0db5c *stdin
user=admin password_hash=a6f62b5131e63fac2e6f1be3e443a12e58e2c5fea002df0924f58eeefb7e81a9 *stdin


In [2]:
!echo "user=Vasya salt=saltAHFG password_hash=$(echo -n saltAHFG%3.1415-2.718182 | openssl sha256 -r)"
!echo "user=Petya salt=saltMSIG password_hash=$(echo -n saltMSIG%sfkjvdjkth | openssl sha256 -r)"
!echo "user=admin salt=saltPQNY password_hash=$(echo -n saltPQNY%3.1415-2.718182 | openssl sha256 -r)"

user=Vasya salt=saltAHFG password_hash=0c9ce37e04e94dc13f16304a93b21e7f2c44ca32d6c26fbea3375ea85263aaa0 *stdin
user=Petya salt=saltMSIG password_hash=df9c27cc066b36be6dc73a39f03e73ec4996b378bfff562421e53bf85f3a99c5 *stdin
user=admin salt=saltPQNY password_hash=8af9bf88fbe91b010c37d5065c90935c5bb51f5e2898bd92a7235581bd0ccb36 *stdin


## Симметричное шифрование

Используется для шифрования больших объемов текста. Для шифрования и расшифровки используется общий секрет.

### Шифроблокноты

Это самый надежный из симметричных шифров: генерируется случайная последовательность большой длины и становится ключом.

https://en.wikipedia.org/wiki/One-time_pad

In [6]:
import random
import base64


def xor(x, y):
    return bytes(a ^ b for a, b in zip(x, y))


print("both Alice and Bob")
common_secret = bytes(random.randint(0, 255) for i in range(35))  # на самом деле тут стоило бы исплользовать более надежный генератор случайных чисел 
print("Содержимое шифроблокнота:", base64.b64encode(common_secret))

print("Alice → ")
plain_text = b"there are several spy secrets here"
print("Текст, который хотим зашифровать (Алиса хочет отправить его Бобу):", plain_text)
cipher_text = xor(plain_text, common_secret)
print("Шифротекст:", base64.b64encode(cipher_text))

print(" → Bob")
recovered_plain_text = xor(cipher_text, common_secret)
print("Текст, который получил Боб: ", recovered_plain_text)


both Alice and Bob
Содержимое шифроблокнота: b'r4tm+LhxMVMjBuIdgcJiO/gYroVG/tofkhLvPf3pwskv1Fg='
Alice → 
Текст, который хотим зашифровать (Алиса хочет отправить его Бобу): b'there are several spy secrets here'
Шифротекст: b'2+MDit1RUCFGJpF496cQWpQ43fU/3ql68WCKSY7JqqxdsQ=='
 → Bob
Текст, который получил Боб:  b'there are several spy secrets here'


Но есть очевидный минус: общий секрет должет быть размера не меньшего, чем весь объем отправляемых данных.

## Блочное шифрование

По сути пара функций: `output_block = E(input_block, secret)` и обратная к ней. `output_block`, `input_block` и `secret` - строки фиксированной длины. Обычно число фигурирующее в названии блочного шифра (AES-256) - это длина ключа в битах.

Казалось бы теперь можно просто зашифровать текст, просто применив функцию блочного шифра поблочно к тексту (называется режимом шифрования ECB), но нет! Иначе есть шанс получить что-то такое :)

<table> 
<tr>
    <th> Исходное изображение </th> <th> Изображение зашифрованное в режиме ECB </th> 

<tr>
    <th> 
        <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/3/35/Tux.svg/300px-Tux.svg.png" width="200" height="200" align="left" alt="Видео с семинара">
    </th>
    <th>
        <img src="https://upload.wikimedia.org/wikipedia/commons/f/f0/Tux_ecb.jpg" width="200" height="200" align="left" alt="Видео с семинара">
    </th>
 
</table>

Чтобы решить эту проблему, используется подход CTR. Его суть в том, чтобы иметь еще счетчик, значение которого будет подмешиваться к секрету при шифровании каждого блока, после чего инкрементироваться. Таким образом, секрет будет "меняться" при каждом новом блоке.

> В реальности схема работает сильно сложнее, но основная идея такая

## Acимметричное шифрование

В симметричном шифровании у отправителя и получателя должен быть общий секрет. А что делать если его нет? Использовать асимметричное шифрование! Оно обычно применяется для обмена некоторой метаинформацией и получения общего секрета.

### Протокол Диффи-Хеллмана

Допустим два агента хотят пообщаться, но у них нет общего ключа и их могу прослушивать. Что делать?

Использовать труднорешаемую задачу :)

Например, это может быть задача дискретного логарифмирования (взятия логарифма в кольце по модулю).

Тогда агенты A и B могут сообща выбрать основание $x$ (через незащищенный канал), потом раздельно выбрать случайные числа $a$, $b$. Возвести $x$ в эти степени и обменяться полученными $x^a$, $x^b$ через незащищенный канал.

Фокус в том, что сейчас люди не умеют по $x$ и $x^a$ находить $a$. Так что $x^a$ передавать безопасно.

А дальше второй фокус: агент A может сделать $(x^b)^a = x^{(a \cdot b)}$, а агент B - $(x^a)^b = x^{(a \cdot b)}$. И получается, что у A и B есть общий секрет. А злоумышленник имея только $x, x^b, x^a$ не может получить $x^{(a \cdot b)}$.

https://ru.wikipedia.org/wiki/Протокол_Диффи_—_Хеллмана

<img src="https://ecwebsitedata.blob.core.windows.net/encryption-consulting-website-data/2025/11/Diffie-Hellman-Key-Exchange-Vs.-RSA.png" width=400 style="background-color:white;"/>

ДИСКЛЕЙМЕР: не стоит использовать самописанные криптографические алгоритмы в реальной жизни, поскольку почти наверное они будут иметь очень большое количество уязвимостей. Рекомендуется всегда использовать проверенные и надежные библиотеки и системы. Под Linux, например, можно использовать `libcrypto`, которая предоставляет много полезных криптографических примитивов (https://github.com/openssl/openssl).